In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf

# Load the data
df = pd.read_csv(r'/content/drive/MyDrive/Datasets_DA/social_media_ab_test_raw.csv')

# Look at the data types and non-null counts
print(df.info())

# Inspect the first few rows to see the mixed date formats and casing
print(df.head(10))

# ==========================================
# STEP 1 - Data Cleaning & Integrity Checks
# ==========================================

# 1. Core Identifiers
missing_users = df['user_id'].isna().sum()
print(f"Initial rows missing user_id: {missing_users}")
df = df.dropna(subset=['user_id'])

# Convert dates using Pandas' mixed format handling
df['exposure_date'] = pd.to_datetime(df['exposure_date'], format='mixed')

# Standardize experiment group strings
df['experiment_group'] = df['experiment_group'].str.lower()

print(f"Total rows before deduplication: {len(df)}")
print(f"Unique users: {df['user_id'].nunique()}")

# 2. Contamination & Deduplication
#If a user saw both Control and Treatment, their behavior is contaminated.
# I must exclude them entirely rather than just keeping their first row.
users_multiple_groups = df.groupby('user_id')['experiment_group'].nunique()
contaminated_users = users_multiple_groups[users_multiple_groups > 1].index
df = df[~df['user_id'].isin(contaminated_users)]

# Now I can safely remove duplicates keeping the first exposure
df = df.sort_values(['user_id', 'exposure_date'])
df = df.drop_duplicates(subset=['user_id'], keep='first')

# 3. Impossible observations

#Drop rows withh impossible engagement
df = df[df['clicks'] <= df['impressions']]

# Actively drop impossible revenue (assuming there were no free trials, etc.)
impossible_revenue_mask = (df['conversions'] > 0) & (df['revenue_eur'] == 0)
df = df[~impossible_revenue_mask]

# 4. Missing Values
# Condition 1: Missing revenue BUT they converted -> Drop
# This is beter than imputing median (which could shrink variance - problematic for t-tests)
revenue_integrity_fail = (df['conversions'] > 0) & (df['revenue_eur'].isna())
df = df[~revenue_integrity_fail]

# Condition 2: Missing revenue and no conversions -> fill with 0
df['revenue_eur'] = df['revenue_eur'].fillna(0)

# Check raw count of missing device_type
missing_device_type_count = df['device_type'].isna().sum()
print(f"Rows missing device type: {missing_device_type_count}")
#Safest category is unknown
df['device_type'] = df['device_type'].fillna('unknown')

# Check raw count of missing countries
missing_country_count = df['country'].isna().sum()
print(f"Rows missing country: {missing_country_count}")
#No such rows

# 5. Cap the outliers - Winsorization
# I condition it on paying users because revenue is zero-inflated and including
# all zeros in Winsorization could make the cap unrealistically low
revenue_99th = df[df['revenue_eur'] > 0]['revenue_eur'].quantile(0.99)
print(f"Capping revenue at the 99th percentile of buyers: €{revenue_99th:.2f}")
df['revenue_eur'] = np.where(df['revenue_eur'] > revenue_99th, revenue_99th, df['revenue_eur'])


# ==========================================
# STEP 2 - KPIs & Aggregations
# ==========================================

# Define the actual user counts per group
users_A = len(df[df['experiment_group'] == 'control'])
users_B = len(df[df['experiment_group'] == 'treatment'])

# SRM check
print("\n--- Check: Sample Ratio Mismatch (SRM) ---")
expected_users_per_group = len(df) / 2
observed_users = [users_A, users_B]
expected_users = [expected_users_per_group, expected_users_per_group]

chi_stat, p_val_srm = chisquare(f_obs=observed_users, f_exp=expected_users)
# I apply strict threshold for the diagnostic test. Otherwise, there is a risk of
# false flagging  ~5 out of 100 A/B tests
if p_val_srm < 0.01:
    print(f"WARNING: SRM detected (p={p_val_srm:.4f}). Traffic split is imbalanced.")
else:
    print(f"No SRM detected (p={p_val_srm:.4f}). Traffic split is balanced.")


# KPI Aggregations (User-Level Metrics)
# I decide to use binary indicators (e.g. clicked or not, rather than the absolute
# number of clicks) to ensure the independence of observations for statistical validity.
# (Several clicks from one user are inherentely not independent).
# Also, I am much more interested in the number of users who clicked, rather than
# how many clicks each user did.

kpi_summary = df.groupby('experiment_group').apply(
    lambda x: pd.Series({
        'Total_Users': len(x),
        'User_CTR': (x['clicks'] > 0).mean(),    # Users who clicked / Total Users
        'CR': (x['conversions'] > 0).mean(),     # Users who converted / Total Users
        'ARPU': x['revenue_eur'].mean()          # Total Revenue / Total Users
    }),
    include_groups=False
).reset_index()

print("\n--- KPI Summary (User-Level) ---")
print(kpi_summary)



# ==========================================
# STEP 3 - Statistical Tests
# ==========================================

print("\n--- Statistical Test Results ---")

# --- Test 1: Z-Test for User Click Rate ---
# -------------------------------------------------------------------------
# STATISTICAL NOTE:
# I run z-tests because these are Bernoulli trials.
# So, the variance is not uknown in this case.
# -------------------------------------------------------------------------


df['clicked'] = np.where(df['clicks'] > 0, 1, 0)

unique_clickers_A = df[df['experiment_group'] == 'control']['clicked'].sum()
unique_clickers_B = df[df['experiment_group'] == 'treatment']['clicked'].sum()

z_stat_ctr, p_val_ctr = proportions_ztest(
    [unique_clickers_B, unique_clickers_A],
    [users_B, users_A]
)
print(f"User Click Rate (Z-Test) p-value: {p_val_ctr:.4f}")


# --- Test 2: Z-Test for User Conversion Rate ---

df['converted'] = np.where(df['conversions'] > 0, 1, 0)

unique_converters_A = df[df['experiment_group'] == 'control']['converted'].sum()
unique_converters_B = df[df['experiment_group'] == 'treatment']['converted'].sum()

z_stat_cr, p_val_cr = proportions_ztest(
    [unique_converters_B, unique_converters_A],
    [users_B, users_A]
)
print(f"User Conversion Rate (Z-Test) p-value: {p_val_cr:.4f}")


# --- Test 3: Welch's T-Test for Revenue (ARPU) ---
# -------------------------------------------------------------------------
# STATISTICAL NOTE:
# Because ARPU is an arithmetic mean, we use Welch's T-Test (which doesn't
# assume equal variance). It accounts for heteroscedasticity.
# Due to the Central Limit Theorem, the T-Test is highly robust to zero-inflation
# and right-skew as long as N is sufficiently large.
# I avoided the Mann-Whitney U test because it tests ranks (medians), not means.
# As a business, we care about total revenue (the mean), and Mann-Whitney would
# ignore the disproportionate impact of our most valuable customers."
# -------------------------------------------------------------------------

# Explicitly define the revenue arrays
revenue_A = df[df['experiment_group'] == 'control']['revenue_eur']
revenue_B = df[df['experiment_group'] == 'treatment']['revenue_eur']

# Run Welch's T-test (equal_var=False)
t_stat, p_val_rev = ttest_ind(revenue_B, revenue_A, equal_var=False, alternative='two-sided')

print(f"ARPU (Welch's T-Test) p-value: {p_val_rev:.4f}")


# ==========================================
# STEP 4 - Visualisation
# ==========================================

# 1. Define the professional annotation function (unchanged)
def add_stat_annotation(ax, p_value, bar_height, bar_gap, text_gap):
    x1, x2 = 0, 1
    y = bar_height + bar_gap
    h = bar_gap / 2

    if p_value < 0.001:
        text = 'p < 0.001 ***'
    elif p_value < 0.01:
        text = f'p = {p_value:.3f} **'
    elif p_value < 0.05:
        text = f'p = {p_value:.3f} *'
    else:
        text = f'p = {p_value:.3f} (ns)'

    ax.text((x1+x2)*.5, y+h+text_gap, text, ha='center', va='bottom', color='black', fontsize=10, fontweight='bold')

# 2. Setup the plot canvas for 3 metrics
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Plot 1: User Click Rate (CTR) ---
sns.barplot(
    data=df,
    x='experiment_group',
    y='clicked',
    ax=axes[0],
    capsize=0.1,
    errorbar=('ci', 95),
    palette=['#A0C4FF', '#4361EE'],
    hue='experiment_group', # Added this
    legend=False            # Added this
)
axes[0].set_title('User Click Rate (95% CI)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Click Rate')

max_ctr = df.groupby('experiment_group')['clicked'].mean().max()
add_stat_annotation(axes[0], p_val_ctr, bar_height=max_ctr, bar_gap=0.02, text_gap=0.005)

# --- Plot 2: User Conversion Rate (CR) ---
sns.barplot(
    data=df,
    x='experiment_group',
    y='converted',
    ax=axes[1],
    capsize=0.1,
    errorbar=('ci', 95),
    palette=['#BEE1E6', '#2A9D8F'],
    hue='experiment_group', # Added this
    legend=False            # Added this
)
axes[1].set_title('User Conversion Rate (95% CI)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Conversion Rate')

max_cr = df.groupby('experiment_group')['converted'].mean().max()
add_stat_annotation(axes[1], p_val_cr, bar_height=max_cr, bar_gap=0.01, text_gap=0.002)

# --- Plot 3: Average Revenue Per User (Global ARPU) ---
sns.barplot(
    data=df,
    x='experiment_group',
    y='revenue_eur',
    ax=axes[2],
    capsize=0.1,
    errorbar=('ci', 95),
    palette=['#FDE2E4', '#E07A5F'],
    hue='experiment_group', # Added this
    legend=False            # Added this
)
axes[2].set_title('ARPU (95% CI)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('ARPU (EUR)')

max_rev = df.groupby('experiment_group')['revenue_eur'].mean().max()
add_stat_annotation(axes[2], p_val_rev, bar_height=max_rev, bar_gap=0.4, text_gap=0.1)

# 3. Final Polish
for ax in axes:
    ax.set_xlabel('Experiment Group')

plt.tight_layout()
plt.savefig('ab_test_3_panel_results.png', dpi=300)
plt.show()


# ==========================================
# STEP 5 - INTERACTION EFFECTS
# ==========================================

#Let's check for the interaction effects
#Start with the device type

print("\n--- Interaction Effects: Device Type ---")
# ---------------------------------------------------------
# Model 1: Logistic Regression for User Click Rate (Binary)
# ---------------------------------------------------------

logit_ctr_model = smf.logit('clicked ~ C(experiment_group) * C(device_type)', data=df).fit(disp=0)

print("\n--- Click Rate Interaction Model ---")
print(logit_ctr_model.summary().tables[1])

#No statistically significant effects

# ---------------------------------------------------------
# Model 2: Logistic Regression for User Conversion Rate (Binary)
# ---------------------------------------------------------

logit_cr_model = smf.logit('converted ~ C(experiment_group) * C(device_type)', data=df).fit(disp=0)

print("\n--- Unconditional Conversion Rate Interaction Model ---")
print(logit_cr_model.summary().tables[1])

#No statistically significant effects

# ---------------------------------------------------------
# Model 3: OLS for ARPU (Continuous)
# ---------------------------------------------------------

ols_revenue_model = smf.ols('revenue_eur ~ C(experiment_group) * C(device_type)', data=df).fit()

print("\n--- ARPU Interaction Model ---")
print(ols_revenue_model.summary().tables[1])

#No statistically significant effects


  experiment_group  Total_Users  User_CTR        CR      ARPU
0          control       2471.0  0.597329  0.108458  4.884464
1        treatment       2463.0  0.676005  0.138449  6.112960
